# Visualizar detecções de galinhas

Este notebook executa o mesmo detector usado pelo comando `chicken-yolo predict` e mostra os quadros originais ao lado das previsões.

**Como usar:** selecione o Python da `.venv` do projeto e execute as células de cima para baixo com o botão ▶ ou `Shift+Enter`. Escolha os parâmetros na etapa 2 e execute a etapa 3 para rodar o modelo. A etapa 4 permite navegar pelos resultados já calculados.

Os vídeos e pesos ficam no armazenamento local. As novas execuções são salvas em `outputs/notebook-.../`, com imagens, configurações e tabelas. O notebook versionado é entregue sem saídas nem imagens incorporadas.

## 1. Preparar o ambiente

Se faltarem dependências, execute na raiz do projeto: `uv sync --frozen --group notebooks`.

No JupyterLab, use o kernel Python da mesma `.venv`. O caminho do Python utilizado aparece abaixo.

In [ ]:
from pathlib import Path
import sys

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file() and (path / "src/chicken_yolo").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Abra este notebook dentro da pasta do projeto.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from chicken_yolo.cli import configure_runtime

configure_runtime(ROOT)

from dataclasses import replace

import ipywidgets as widgets
from IPython.display import HTML, display

from chicken_yolo.config import SlicingConfig, load_inference
from chicken_yolo.preview import preview_table, preview_viewer, run_preview

get_ipython().run_line_magic("matplotlib", "inline")
print("Projeto:", ROOT)
print("Python:", sys.executable)
print("Ambiente pronto. Escolha os parâmetros na próxima etapa.")

## 2. Escolher a entrada e os parâmetros

**Entrada** aceita o caminho de um vídeo, imagem ou pasta no seu computador. O caminho pode ser relativo à raiz do projeto ou absoluto. O exemplo abaixo usa o segundo vídeo da Raiar, caso ele já esteja disponível localmente.

O modo **Comparar direto e SAHI** executa duas configurações sobre os mesmos quadros. SAHI divide a imagem em blocos; pode ajudar com aves pequenas, mas custa mais tempo e também pode gerar erros.

Após alterar os controles, execute a célula da etapa 3. Mover os controles, sozinho, não inicia a inferência.

In [ ]:
style = {"description_width": "160px"}
wide = widgets.Layout(width="95%")

entrada = widgets.Text(
    value="data/raw/raiar-aviario-video-02/video.mp4",
    description="Entrada:", style=style, layout=wide,
)
modelo = widgets.Dropdown(
    options=[("PoultryVision · galinhas", "poultryvision"), ("YOLO11n COCO · aves", "coco")],
    value="poultryvision", description="Modelo:", style=style, layout=wide,
)
pesos = widgets.Text(
    value="", description="Pesos próprios:", style=style, layout=wide,
    placeholder="Opcional: deixe vazio para usar o modelo pré-treinado acima",
)
confianca = widgets.FloatSlider(
    value=.25, min=.05, max=.95, step=.05, readout_format=".2f",
    description="Confiança mínima:", style=style, layout=wide,
    continuous_update=False,
)
resolucao = widgets.Dropdown(
    options=[640, 960, 1280], value=640, description="Resolução:", style=style,
)
modo = widgets.Dropdown(
    options=[("Comparar direto e SAHI", "compare"), ("Somente direto", "standard"),
             ("Somente SAHI", "sliced")],
    value="compare", description="Modo:", style=style, layout=wide,
)
bloco = widgets.Dropdown(
    options=[128, 256, 384, 512], value=256, description="Bloco SAHI (px):", style=style,
)
intervalo = widgets.BoundedFloatText(
    value=2., min=.1, max=120., step=.5, description="Intervalo (s):", style=style,
)
limite = widgets.BoundedIntText(
    value=6, min=1, max=24, description="Máximo de quadros:", style=style,
)
display(widgets.VBox([
    entrada, modelo, pesos, confianca, resolucao, modo, bloco, intervalo, limite,
    widgets.HTML("<p>Confiança é um limiar do detector, não uma medida de acurácia. "
                 "Comece com poucos quadros para testar.</p>"),
]))

## 3. Executar o modelo

**Esta célula roda a detecção de verdade.** Ela lê a entrada escolhida, prepara os pesos locais (ou os baixa na primeira utilização), processa os quadros e salva uma nova execução. Aguarde a indicação de término antes de executar a etapa 4.

Reexecutar esta célula gera uma nova análise com os valores atuais dos controles. Os vídeos originais não são modificados.

In [ ]:
preview = None
base, _ = load_inference(ROOT / "configs/inference.yaml")
config = replace(
    base,
    model=modelo.value,
    weights=pesos.value.strip() or None,
    confidence=confianca.value,
    imgsz=resolucao.value,
    sample_seconds=intervalo.value,
    slicing=SlicingConfig(height=bloco.value, width=bloco.value, overlap=.2, merge_iou=.5),
)
modes = ("standard", "sliced") if modo.value == "compare" else (modo.value,)
print("Executando o detector. SAHI pode demorar mais na CPU.")
preview = run_preview(
    ROOT, entrada.value.strip().strip('"'), config,
    max_frames=limite.value, modes=modes,
)
print(f"Execução encerrada: {len(preview.frames)} quadros e {len(preview.results)} modos.")
print("Resultados:", preview.output)
print("Consulte os estados de processamento na tabela da etapa 5.")

## 4. Visualizar os quadros lado a lado

Execute esta célula e mova o controle **Quadro**. A imagem original aparece à esquerda; as previsões dos modos escolhidos aparecem ao lado.

**Trocar o quadro aqui apenas visualiza a execução anterior.** Para testar outros parâmetros, altere os controles da etapa 2, reexecute a etapa 3 e depois esta célula.

In [ ]:
if globals().get("preview") is None:
    raise RuntimeError("Execute a etapa 3 com sucesso antes de visualizar.")
display(preview_viewer(preview))

## 5. Conferir a tabela e os arquivos

Cada linha corresponde a um quadro e modo. `caixas_previstas` é a quantidade de previsões do detector, não uma contagem validada de galinhas. Uma caixa pode marcar um grupo ou uma estrutura, e várias aves podem passar sem detecção. Não some quadros como animais únicos.

`valido` indica que o processamento técnico terminou; não comprova acerto do modelo. Falhas mantêm contagens indisponíveis na tabela. Este notebook não treina o modelo nem mede precisão, recall ou mAP.

In [ ]:
if globals().get("preview") is None:
    raise RuntimeError("Execute a etapa 3 antes de consultar os resultados.")
display(preview_table(preview))
print("Pasta da execução:", preview.output)
print("Comparativo:", preview.output / "comparison.csv")
print("Frames originais:", preview.output / "frames")
print("As imagens com detecções estão em <modo>/annotated/.")